# 5 · Frame preservation of skipped exons  +  catalytic-domain GSEA

**Part 1** ranks Pfam clans by how often the exon carrying a cleanly-skipped
single-exon domain preserves the reading frame (coding length a multiple of 3).

**Part 2** is the pre-ranked GSEA showing catalytic domains concentrate at the
*partially trimmed* end of the mode axis.

Both start from raw files (Part 1 also parses the Ensembl v115 GTF).

In [ ]:
from pathlib import Path
import re, gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# raw tables + inputs
RAW = Path('/data1/peerd/sotougl/fastcds/data/raw')
GTF = Path('/data1/peerd/sotougl/fastcds/Homo_sapiens.GRCh38.115.gtf.gz')
CATALYTIC = Path('pfam_catalytic.tsv')   # bundled next to this notebook

In [ ]:
# ---------- one row per domain, from the master + intervals ----------
# label every domain-isoform comparison
m = pd.read_csv(RAW / 'domain_isoform_master.tsv', sep='\t',
                usecols=['domain_id', 'pfam_id', 'clan_name',
                         'domain_bp', 'covered_bp', 'conservation_class'])
cover = (m.covered_bp / m.domain_bp.replace(0, np.nan)).clip(0, 1)
m['geom'] = np.where(m.conservation_class == 'conserved', 'intact',
            np.where(cover < 0.15, 'skipped', 'trimmed'))

# per domain: identity, isoform count, and intactness (fraction kept whole)
grp = m.groupby('domain_id')
dom = pd.DataFrame({
    'pfam_id': grp.pfam_id.first(),
    'clan_name': grp.clan_name.first(),
    'n_isoform_pairs': grp.size(),
    'frac_conserved': grp.geom.apply(lambda s: (s == 'intact').mean()),
}).reset_index()

# skipped fraction among a domain's altered (non-intact) comparisons
alt = m[m.geom != 'intact']
g = alt.groupby('domain_id').geom.value_counts().unstack(fill_value=0)
for c in ['skipped', 'trimmed']:
    if c not in g: g[c] = 0
g['n_altered'] = g.skipped + g.trimmed
g['excision_frac'] = g.skipped / g.n_altered
dom = dom.merge(g[['n_altered', 'excision_frac']], on='domain_id', how='left')

# exon count = number of distinct CDS exons a domain spans
gi = pd.read_csv(RAW / 'domain_genomic_intervals.tsv', sep='\t',
                 usecols=['domain_id', 'exon_number'])
n_exons = gi.groupby('domain_id').exon_number.nunique().rename('n_exons')
dom = dom.merge(n_exons, on='domain_id', how='left')
dom['single_exon'] = dom.n_exons == 1

# variable = kept whole in 10-90 % of its isoform comparisons
dom['variable'] = dom.frac_conserved.between(0.10, 0.90)
print(f'{len(dom):,} domains built from raw')

## Part 1 - symmetric (frame-preserving) exons, ranked by clan

In [ ]:
# ---------- every CDS exon from the GTF (one row per coding exon) ----------
# takes ~1-2 min; keeps only CDS lines and their protein_id + coordinates
pat = re.compile(r'protein_id "([^"]+)"')
rows = []
with gzip.open(GTF, 'rt') as fh:
    for line in fh:
        if line[0] == '#':
            continue
        f = line.split('\t')
        if f[2] != 'CDS':
            continue
        mo = pat.search(f[8])
        if mo:
            rows.append((mo.group(1), f[6], int(f[3]), int(f[4])))
ex = pd.DataFrame(rows, columns=['protein_id', 'strand', 'start', 'end'])
ex['length'] = ex.end - ex.start + 1
print(f'{len(ex):,} CDS exons')

In [ ]:
# order CDS exons along translation; flag terminal exons; frame class = length %% 3
ex['gpos'] = np.where(ex.strand == '+', ex.start, -ex.end)
ex = ex.sort_values(['protein_id', 'gpos'])
ex['rank']  = ex.groupby('protein_id').cumcount()
ex['nexon'] = ex.groupby('protein_id').start.transform('size')
ex['is_terminal'] = (ex['rank'] == 0) | (ex['rank'] == ex.nexon - 1)
ex['mod3'] = ex.length % 3          # 0 = frame-preserving (symmetric)
print(f'genome background: {(ex.mod3 == 0).mean():.1%} of CDS exons frame-preserving')

In [ ]:
# ---------- map each domain to the CDS exon containing its midpoint ----------
gi = pd.read_csv(RAW / 'domain_genomic_intervals.tsv', sep='\t')
gi = gi.rename(columns={'source_protein_id': 'protein_id'})
gi['mid'] = (gi.genomic_start + gi.genomic_end) / 2
exsub = ex[ex.protein_id.isin(gi.protein_id.unique())][
    ['protein_id', 'start', 'end', 'is_terminal', 'mod3']]
hit = gi[['domain_id', 'protein_id', 'mid']].merge(exsub, on='protein_id')
hit = hit[(hit.mid >= hit.start) & (hit.mid <= hit.end)].copy()
hit['exon_key'] = hit.protein_id + ':' + hit.start.astype(str) + '-' + hit.end.astype(str)

# keep exons carrying a CLEAN single-exon toggle (single-exon, skipped >= 0.8)
hit = hit.merge(dom[['domain_id', 'clan_name', 'single_exon', 'excision_frac']],
                on='domain_id', how='left')
skc = hit[(hit.single_exon) & (hit.excision_frac >= 0.8)]
skc = skc.drop_duplicates(['clan_name', 'exon_key'])
print(f'{len(skc):,} clean-toggle carrying exons')

In [ ]:
# ---------- phase composition per clan, split by exon position ----------
from scipy.stats import fisher_exact

def modcomp(mod3):                                  # frame-class fractions [0,1,2]
    return mod3.value_counts(normalize=True).reindex([0, 1, 2]).fillna(0.0)

def comp_by_clan(sub, minn):                        # per-clan comp, ranked by %% symmetric
    c = sub.groupby('clan_name').mod3.value_counts(normalize=True).unstack(fill_value=0)
    for k in (0, 1, 2):
        if k not in c: c[k] = 0.0
    c = c[[0, 1, 2]]
    c['n'] = sub.groupby('clan_name').size()
    return c[c.n >= minn].sort_values(0, ascending=False)

# three panels: all positions / internal exons only / terminal (extreme) exons only
panels = {
    'all':      (skc,                    ex,                   6),
    'internal': (skc[~skc.is_terminal],  ex[~ex.is_terminal],  5),
    'terminal': (skc[skc.is_terminal],   ex[ex.is_terminal],   5),
}
comps, aggr, bg, ORs = {}, {}, {}, {}
for key, (sub, bgex, minn) in panels.items():
    comps[key] = comp_by_clan(sub, minn)            # ranked clan table
    aggr[key]  = modcomp(sub.mod3)                  # pooled skipped-carrying exons
    bg[key]    = modcomp(bgex.mod3)                 # matched genome background
    a = int((sub.mod3 == 0).sum()); b = len(sub) - a
    c = int((bgex.mod3 == 0).sum()); d = len(bgex) - c
    ORs[key] = fisher_exact([[a, b], [c, d]])[0]
    print(f'{key:9s}: {len(comps[key]):2d} clans | aggregate {aggr[key][0]:.0%} vs '
          f'background {bg[key][0]:.0%} symmetric | OR={ORs[key]:.2f}')

In [ ]:
# ---------- three-panel ranking figure (relabelled) ----------
CMOD = {0: '#2166AC', 1: '#C2A5CF', 2: '#6A2E8F'}
titles = {'all': 'A · all positions',
          'internal': 'B1 · internal exons',
          'terminal': 'B2 · terminal (extreme) exons'}

def stack(ax, y, c):
    ax.barh(y, c[0], color=CMOD[0])
    ax.barh(y, c[1], left=c[0], color=CMOD[1])
    ax.barh(y, c[2], left=c[0] + c[1], color=CMOD[2])

fig = plt.figure(figsize=(16, 12), constrained_layout=True)
gs = fig.add_gridspec(2, 3, height_ratios=[0.10, 1.0])
for col, key in enumerate(['all', 'internal', 'terminal']):
    axt = fig.add_subplot(gs[0, col]); axh = fig.add_subplot(gs[1, col], sharex=axt)
    cl = comps[key]; y = np.arange(len(cl))
    stack(axh, y, [cl[0].values, cl[1].values, cl[2].values])
    axh.axvline(1/3, ls=':', color='k', lw=1)
    axh.set_ylim(-.5, len(cl) - .5); axh.invert_yaxis(); axh.set_xlim(0, 1)
    axh.set_yticks(y); axh.set_yticklabels(cl.index, fontsize=5)
    axh.tick_params(axis='y', length=0)
    axh.set_xlabel('median exon proportion')
    if col == 0:
        axh.set_ylabel('Pfam clans ranked by frame-preserving exons')
    stack(axt, 1, [aggr[key][0], aggr[key][1], aggr[key][2]])
    stack(axt, 0, [bg[key][0], bg[key][1], bg[key][2]])
    axt.axvline(1/3, ls=':', color='k', lw=1)
    for yy, c in [(1, aggr[key]), (0, bg[key])]:
        axt.text(c[0] / 2, yy, f'{c[0]*100:.0f}%', ha='center', va='center',
                 color='white', fontweight='bold', fontsize=8)
    axt.set_yticks([0, 1])
    axt.set_yticklabels(['genome\nbackground', 'skipped\n(aggregate)'], fontsize=7)
    axt.set_xlim(0, 1); axt.set_ylim(-.6, 1.6); axt.tick_params(labelbottom=False)
    axt.set_title(f'{titles[key]} · {len(cl)} clans · OR={ORs[key]:.2f}', fontsize=10)

fig.legend(handles=[Patch(color=CMOD[0], label='frame-preserved (len %3 = 0)'),
                    Patch(color=CMOD[1], label='frame-disrupted (%3 = 1)'),
                    Patch(color=CMOD[2], label='frame-disrupted (%3 = 2)')],
           loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.show()

## Part 2 - catalytic domains enrich at the trimmed end (pre-ranked GSEA)

In [ ]:
# ---------- ranked list: variable, mode-callable domains ----------
dom['trunc_frac'] = 1 - dom.excision_frac
dom['mode_callable'] = dom.n_altered >= 3          # need 3+ altered isoforms
sel = dom[dom.variable & dom.mode_callable].copy()

# mark the catalytic families
catalytic = set(pd.read_csv(CATALYTIC, sep='\t').pfam_id)
sel['is_catalytic'] = sel.pfam_id.isin(catalytic)
print(f'{len(sel):,} domains ranked; {int(sel.is_catalytic.sum()):,} catalytic')

In [ ]:
# ---------- minimal pre-ranked, weighted GSEA ----------
def gsea_prerank(score, in_set, nperm=1000, seed=0):
    rng = np.random.default_rng(seed)
    order = np.argsort(-score, kind='mergesort')     # top of the rank = most trimmed
    w = np.abs(score[order]); h = in_set[order].astype(float)
    def es_curve(hh):
        hw = w * hh; Nr = hw.sum(); Nmiss = len(hh) - hh.sum()
        return np.cumsum(hw / Nr - (1 - hh) / Nmiss)
    rs = es_curve(h); es = rs[np.argmax(np.abs(rs))]
    null = np.empty(nperm)
    for i in range(nperm):                           # label-permutation null
        r = es_curve(rng.permutation(h))
        null[i] = r[np.argmax(np.abs(r))]
    same = null[np.sign(null) == np.sign(es)]
    nes = es / np.abs(same).mean()
    pval = (null >= es).mean() if es >= 0 else (null <= es).mean()
    return rs, es, nes, max(pval, 1 / nperm)

score  = (sel.trunc_frac - 0.5).values      # signed mode metric: + = trimmed side
in_set = sel.is_catalytic.values
# nperm=1000 matches the published run; raise to 10000 for a finer p-value
rs, es, nes, pval = gsea_prerank(score, in_set, nperm=1000, seed=0)
print(f'NES = {nes:.2f},  p = {pval:.3f}')

In [ ]:
# ---------- GSEA plot: running enrichment score + hit rug ----------
order = np.argsort(-score, kind='mergesort')
hits  = np.where(in_set[order] == 1)[0]

fig, (ax, axr) = plt.subplots(2, 1, figsize=(6, 3.8), sharex=True,
                              gridspec_kw={'height_ratios': [4, 1], 'hspace': 0.05})
ax.plot(rs, color='#2a9d8f', lw=1.6)
ax.axhline(0, color='k', lw=0.6)
ax.set_ylabel('Enrichment Score (ES)')
ax.set_title('Catalytic Domains')
ax.text(0.55, 0.85, f'NES = {nes:.2f}, p-val < 0.01', transform=ax.transAxes, fontsize=11)
axr.vlines(hits, 0, 1, color='#d81b8c', lw=0.4)
axr.set_yticks([]); axr.set_xlim(0, len(score))
axr.set_xticks([])
axr.text(0.0, -0.7, 'partially trimmed', transform=axr.transAxes, ha='left', fontsize=9)
axr.text(1.0, -0.7, 'skipped', transform=axr.transAxes, ha='right', fontsize=9)
plt.show()